In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.terminal.shortcuts.filters import pass_through

In [2]:
df = pd.read_csv('importation-v1(COMPLETED).csv')

In [3]:
df.head()

,year,port,country,HS,weight-kg,worth-usd,HS-Short-Name,distance,delivery-days,s-cost,tariff,VAT,Tariff-Amount-USD,VAT-Amount-USD,Total-Import-Cost-USD,Profit-Margin-Pct,Cost-per-kg-USD,Profit-per-kg-USD,Total-Import-Profit-USD
0,1403,shaeed rajaii,China,98870315,11179099,176091909,Vehicle/Chassis/Tractor Parts,10500,30,1676864.85,40%,9%,70436763.6,22187580.53,270393118.0,25%,24.19,6.05,67598279.50
1,1403,shaeed rajaii,China,98871182,20632931,162439389,Vehicle/Chassis/Tractor Parts,10500,30,3094939.65,40%,9%,64975755.6,20467363.01,250977447.3,25%,12.16,3.04,62744361.82
2,1403,shaeed rajaii,China,98871183,20275247,140716265,Vehicle/Chassis/Tractor Parts,10500,30,3041287.05,40%,9%,56286506.0,17730249.39,217774307.4,25%,10.74,2.68,54443576.86
3,1403,shaeed rajaii,China,87032290,14735380,124830304,Passenger Vehicles,10500,30,2210307.00,45%,9%,56173636.8,16290354.67,199504602.5,25%,13.54,3.38,49876150.62
4,1403,shaeed rajaii,China,98870121,14375399,113825856,Vehicle/Chassis/Tractor Parts,10500,30,2156309.85,40%,9%,45530342.4,14342057.86,175854566.1,25%,12.23,3.06,43963641.53


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7113 entries, 0 to 7112
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     7113 non-null   int64  
 1   port                     7113 non-null   object 
 2   country                  7113 non-null   object 
 3   HS                       7113 non-null   int64  
 4   weight-kg                7113 non-null   int64  
 5   worth-usd                7113 non-null   int64  
 6   HS-Short-Name            7113 non-null   object 
 7   distance                 7113 non-null   int64  
 8   delivery-days            7113 non-null   int64  
 9   s-cost                   7113 non-null   float64
 10  tariff                   7113 non-null   object 
 11  VAT                      7113 non-null   object 
 12  Tariff-Amount-USD        7113 non-null   float64
 13  VAT-Amount-USD           7113 non-null   float64
 14  Total-Import-Cost-USD   

In [5]:
df.drop(columns=['year','port','HS', 'tariff', 'VAT', 'Profit-Margin-Pct', 'Profit-per-kg-USD', 'distance', 'delivery-days','s-cost', 'Tariff-Amount-USD', 'VAT-Amount-USD', 'Total-Import-Cost-USD', 'Cost-per-kg-USD'], inplace=True)

In [6]:
list(df.columns)

['country',
 'weight-kg',
 'worth-usd',
 'HS-Short-Name',
 'Total-Import-Profit-USD']

In [7]:
df['HS-Short-Name'].unique()

array(['Vehicle/Chassis/Tractor Parts', 'Passenger Vehicles',
       'Mineral Fuels & Oils', 'Electrical Equipment',
       'Iron & Steel Products', 'Semiconductors & Solar Cells',
       'Plastics & Polymers', 'Paper & Paperboard', 'Compressors & Pumps',
       'General Goods', 'Machines & Mechanical Appliances',
       'Rubber & Tires', 'Industrial Machinery',
       'Articles of Iron & Steel', 'Vehicle Parts & Accessories',
       'Organic Chemicals', 'Chemical Products',
       'Telecom & Display Equipment'], dtype=object)

In [8]:
y_data = df['Total-Import-Profit-USD']
x_data = df.drop(columns=['Total-Import-Profit-USD'])

In [9]:
y_data

0       67598279.50
1       62744361.82
2       54443576.86
3       49876150.62
4       43963641.53
           ...     
7108         357.04
7109         284.82
7110         193.62
7111         191.56
7112         130.35
Name: Total-Import-Profit-USD, Length: 7113, dtype: float64

In [10]:
x_data

,country,weight-kg,worth-usd,HS-Short-Name
0,China,11179099,176091909,Vehicle/Chassis/Tractor Parts
1,China,20632931,162439389,Vehicle/Chassis/Tractor Parts
2,China,20275247,140716265,Vehicle/Chassis/Tractor Parts
3,China,14735380,124830304,Passenger Vehicles
4,China,14375399,113825856,Vehicle/Chassis/Tractor Parts
...,...,...,...,...
7108,Germany,500,1046,Plastics & Polymers
7109,Germany,9,908,Electrical Equipment
7110,Germany,9,617,Electrical Equipment
7111,Germany,3,611,Industrial Machinery


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
train_x, test_x, train_y, test_y = train_test_split(x_data, y_data, test_size=0.2, random_state=42)
print("X_train shape:", train_x.shape)
print("X_test shape:", test_x.shape)

X_train shape: (5690, 4)
X_test shape: (1423, 4)


In [13]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [14]:
categorical_features = ["HS-Short-Name", "country"]
numerical_features = ["weight-kg", "worth-usd"]

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [16]:
train_x.head()

,country,weight-kg,worth-usd,HS-Short-Name
1755,China,47809,345695,General Goods
3207,China,1350,3630,General Goods
443,China,195123,4092914,General Goods
2807,China,22000,37532,General Goods
3240,China,10000,2481,General Goods


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [18]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', RandomForestRegressor(random_state=42))])
pipeline.fit(train_x, train_y)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
train_x

,country,weight-kg,worth-usd,HS-Short-Name
1755,China,47809,345695,General Goods
3207,China,1350,3630,General Goods
443,China,195123,4092914,General Goods
2807,China,22000,37532,General Goods
3240,China,10000,2481,General Goods
...,...,...,...,...
3772,UAE,13000,1999165,Organic Chemicals
5191,UAE,68000,63162,General Goods
5226,UAE,37600,56569,Mineral Fuels & Oils
5390,UAE,29478,34202,Iron & Steel Products


In [20]:
# from sklearn.metrics import mean_absolute_error, r2_score
#
# # 1. Ask the model to predict profits on the test set
# predictions = pipeline.predict(test_x)
#
# # 2. Calculate metrics
# mae = mean_absolute_error(test_y, predictions)
# r2 = r2_score(test_y, predictions)
#
# print(f"Model Accuracy (R-squared): {r2*100:.2f}%")
# print(f"Average Error (MAE): ${mae:,.2f}")

In [21]:
df['profit_per_worth'] = df['Total-Import-Profit-USD'] / df['worth-usd']

rate_check = df.groupby(['country', 'HS-Short-Name'])['profit_per_worth'].agg(['mean', 'std', 'count'])
rate_check = rate_check.sort_values('std', ascending=False)
print(rate_check)

                                              mean       std  count
country HS-Short-Name                                              
Germany General Goods                     0.361029  0.211515    117
China   Mineral Fuels & Oils              0.328118  0.048639     16
Germany Mineral Fuels & Oils              0.320472  0.035476      3
India   General Goods                     0.341403  0.028560    303
UAE     Mineral Fuels & Oils              0.308419  0.021535     16
...                                            ...       ...    ...
        Telecom & Display Equipment       0.313710  0.000241     17
Germany Compressors & Pumps               0.313938  0.000214      8
        Telecom & Display Equipment       0.313526       NaN      1
India   Machines & Mechanical Appliances  0.316515       NaN      1
        Vehicle/Chassis/Tractor Parts     0.383526       NaN      1

[68 rows x 3 columns]
